### Phase 5: Olist Data Engineering Pipeline (SQL Version)


In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

base_path = "/Volumes/filestore/filestore/olist/"

In [0]:
customers = spark.read.option("header", "true").csv(base_path + "olist_customers_dataset.csv")
orders = spark.read.option("header", "true").csv(base_path + "olist_orders_dataset.csv")
order_items = spark.read.option("header", "true").csv(base_path + "olist_order_items_dataset.csv")
products = spark.read.option("header", "true").csv(base_path + "olist_products_dataset.csv")
translation = spark.read.option("header", "true").csv(base_path + "product_category_name_translation.csv")

In [0]:
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")

### TASK 1: Top 3 Customers per City

In [0]:
%sql
SELECT *
FROM (
    SELECT 
        c.customer_city AS city,
        o.customer_id,
        SUM(oi.price) AS total_spend,
        RANK() OVER (
            PARTITION BY c.customer_city 
            ORDER BY SUM(oi.price) DESC
        ) AS rank
    FROM customers c
    JOIN orders o 
        ON c.customer_id = o.customer_id
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY c.customer_city, o.customer_id
) t
WHERE rank <= 3;

city,customer_id,total_spend,rank
abadia dos dourados,9e01f714a2b3b8962c222cf2b74c20dc,199.0,1
abadia dos dourados,a23e3f9a2b656b23b7e52075964b42cd,120.0,2
abadia dos dourados,f11eb8f0b8b87510a93e3e1aa10b0ade,39.9,3
abadiania,576d71ddb21b21763cfedce73b902180,949.99,1
abaete,d47c8bb51df6f716e196ecd6cd5d2c09,449.0,1
abaete,ff0d62f8be4c098e6306f39bc6ebded4,225.9,2
abaete,5371894984937a27cf40c7d20699a786,208.9,3
abaetetuba,c7eb06383ae604616cb4c9d36fe6745e,1500.0,1
abaetetuba,367fd22f1e994de87cf447d29c167a1f,797.6,2
abaetetuba,7727e2cc9ec428ad3173cc812ce1781b,580.27,3


###  TASK 2: Running Total of Sales

In [0]:
%sql
SELECT 
    order_date,
    daily_sales,
    SUM(daily_sales) OVER (ORDER BY order_date) AS running_total
FROM (
    SELECT 
        DATE(o.order_purchase_timestamp) AS order_date,
        SUM(oi.price) AS daily_sales
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY DATE(o.order_purchase_timestamp)
) t;

order_date,daily_sales,running_total
2016-09-04,72.89,72.89
2016-09-05,59.5,132.39
2016-09-15,134.97,267.36
2016-10-02,100.0,367.36
2016-10-03,463.48,830.84
2016-10-04,9940.96,10771.8
2016-10-05,8343.25,19115.05
2016-10-06,7960.51,27075.559999999998
2016-10-07,7228.05,34303.61
2016-10-08,8441.849999999999,42745.46


### TASK 3: Top Products per Category

In [0]:
%sql
SELECT *
FROM (
    SELECT 
        p.product_category_name AS category,
        oi.product_id,
        SUM(oi.price) AS total_sales,
        DENSE_RANK() OVER (
            PARTITION BY p.product_category_name 
            ORDER BY SUM(oi.price) DESC
        ) AS rank
    FROM order_items oi
    JOIN products p 
        ON oi.product_id = p.product_id
    GROUP BY p.product_category_name, oi.product_id
) t;

category,product_id,total_sales,rank
null,5a848e4ab52fd5445cdc07aab1c40e48,24229.029999999984,1
null,eed5cbd74fac3bd79b7c7ec95fa7507d,9945.0,2
null,b1d207586fca400a2370d50a9ba1da98,7152.0,3
null,76d1a1a9d21ab677a61c3ae34b1b352f,5712.64,4
null,ad88641611c35ebd59ecda07a9f17099,4515.330000000001,5
null,3b60d513e90300a4e9833e5cda1f1d61,4393.33,6
null,4c50dcc50f1512f46096d6ef0142c4a9,3980.0,7
null,17823ffd2de8234f0e885a71109613a4,2969.89,8
null,0e030462875259ec0cb868f7ecf1fd5e,2740.0,9
null,b36f3c918c91478c4559160022d3f14e,2550.0,10


### TASK 4: Customer Lifetime Value (CLV)

In [0]:
%sql
SELECT 
    o.customer_id,
    SUM(oi.price) AS total_spend
FROM orders o
JOIN order_items oi 
    ON o.order_id = oi.order_id
GROUP BY o.customer_id;

customer_id,total_spend
cf8ffeddf027932e51e4eae73b384059,179.0
241e78de29b3090cfa1b5d73a8130c72,113.0
e1862648f338ecaf4242e8e2b59126ca,37.99
3c628393675b42c6b5ef89461f68ecef,359.8
328d7a69cb9cbaf088eed3ed778804bb,90.0
aa601b3c45980c0918042d5ca7a25054,49.99
d4c5a2a1316f738e72e179d6bb3b6d8f,96.0
ae8db0691449a44352e7d535ddf78c5e,109.9
4d225460f83ea2b1ac705ad1d5eef02f,89.99
af5b08db5f1a31a3b6856a8a6dc33167,39.99


### TASK 5: Customer Segmentation

In [0]:
%sql
SELECT 
    customer_id,
    total_spend,
    CASE 
        WHEN total_spend > 10000 THEN 'Gold'
        WHEN total_spend BETWEEN 5000 AND 10000 THEN 'Silver'
        ELSE 'Bronze'
    END AS segment
FROM (
    SELECT 
        o.customer_id,
        SUM(oi.price) AS total_spend
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY o.customer_id
) t;

customer_id,total_spend,segment
cf8ffeddf027932e51e4eae73b384059,179.0,Bronze
241e78de29b3090cfa1b5d73a8130c72,113.0,Bronze
e1862648f338ecaf4242e8e2b59126ca,37.99,Bronze
3c628393675b42c6b5ef89461f68ecef,359.8,Bronze
328d7a69cb9cbaf088eed3ed778804bb,90.0,Bronze
aa601b3c45980c0918042d5ca7a25054,49.99,Bronze
d4c5a2a1316f738e72e179d6bb3b6d8f,96.0,Bronze
ae8db0691449a44352e7d535ddf78c5e,109.9,Bronze
4d225460f83ea2b1ac705ad1d5eef02f,89.99,Bronze
af5b08db5f1a31a3b6856a8a6dc33167,39.99,Bronze


### TASK 6: Final Reporting Table

In [0]:
%sql
SELECT 
    c.customer_id,
    c.customer_city AS city,
    SUM(oi.price) AS total_spend,
    COUNT(DISTINCT o.order_id) AS total_orders,
    CASE 
        WHEN SUM(oi.price) > 10000 THEN 'Gold'
        WHEN SUM(oi.price) BETWEEN 5000 AND 10000 THEN 'Silver'
        ELSE 'Bronze'
    END AS segment
FROM customers c
JOIN orders o 
    ON c.customer_id = o.customer_id
JOIN order_items oi 
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.customer_city;

customer_id,city,total_spend,total_orders,segment
6ca4c791dfb84fbc05cbe68d7d7a95be,sao paulo,140.0,1,Bronze
08ebd000e60012404de32cacee764799,patos de minas,46.4,1,Bronze
d18cd4e15b74421f0bedb7bc0db2ed2c,itatiba,89.5,1,Bronze
3dfbf844abeffee1631f66fced77dbef,jundiai,33.49,1,Bronze
86e38a0e0d9692b4d867a678af61ffa5,rio claro,379.9,1,Bronze
5b43ef89a22571e7993b5179c5ab40a9,salto do lontra,519.6,1,Bronze
c8ece683fdd660e0bef8c967d4a275da,cascavel,54.9,1,Bronze
94595ccee21b11325952776b516a0345,rio de janeiro,330.99,1,Bronze
8eb90c8fee300fde9364d8128df008fc,sao paulo,56.89,1,Bronze
f03c9c80de7a0a901cd92d8507717007,sao vicente,195.0,1,Bronze
